# Phonological Distance Analysis

Investigates *why* ASR models (Qwen, Gemini) underperform on Nigerian and Chinese-accented English by measuring acoustic distance in XLS-R 300M embedding space.

**Pipeline stages** (run once each, then reuse outputs):
1. Extract 16kHz mono WAVs from the HF dataset
2. Transcribe with Whisper large-v3 (word-level timestamps)
3. Compute per-word DTW distances vs. American English reference speakers
4. *(This notebook)* Phonological category grouping + visualization

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, re

ROOT  = Path(os.getcwd()).parent
OUT   = ROOT / "results" / "phonological_distance"
DIST  = OUT  / "distances" / "word_distances.csv"

ACCENT_ORDER  = ["Nigerian", "Chinese", "Indian", "British", "American"]
ACCENT_COLORS = {
    "Nigerian": "#d62728",
    "Chinese":  "#ff7f0e",
    "Indian":   "#2ca02c",
    "British":  "#1f77b4",
    "American": "#9467bd",
}

## Stage 1 — Extract WAV files
Reads audio bytes from the cached HuggingFace Arrow file and writes 16kHz mono WAVs.

In [ ]:
# Run once — skips if wav files already exist
!cd .. && /opt/homebrew/bin/python3.10 scripts/phonological_distance_pipeline.py --stage extract

In [ ]:
# Verify: show speaker manifest
with open(OUT / "speakers.json") as f:
    speakers = json.load(f)

df_sp = pd.DataFrame(speakers)
print(df_sp["accent"].value_counts().to_string())
print(f"\nTotal: {len(df_sp)} speakers")

## Stage 2 — Transcribe with Whisper

Uses `openai/whisper-large-v3` via the HuggingFace pipeline with `return_timestamps='word'`.
Output: per-speaker TSV files with `word \t start \t end` (filtered to script vocabulary).

> **Spot-check** a few Nigerian and Chinese speakers after this runs to verify timestamps look reasonable.

In [ ]:
# Skips already-transcribed speakers
!cd .. && /opt/homebrew/bin/python3.10 scripts/phonological_distance_pipeline.py --stage transcribe

In [ ]:
# Spot-check: show word coverage per speaker
import csv
from pathlib import Path

ts_dir = OUT / "timestamps"
rows = []
for p in sorted(ts_dir.glob("*.tsv")):
    with open(p, newline="") as f:
        n = sum(1 for _ in csv.DictReader(f, delimiter="\t"))
    sp_id = p.stem
    accent = next((s["accent"] for s in speakers if s["speaker_id"] == sp_id), "?")
    rows.append({"speaker": sp_id, "accent": accent, "n_words": n})

df_ts = pd.DataFrame(rows).sort_values(["accent","speaker"])
print(df_ts.to_string(index=False))
print("\nMean words by accent:")
print(df_ts.groupby("accent")["n_words"].agg(["mean","min","max"]).round(1))

In [ ]:
# Show first few words for a Nigerian and Chinese speaker
for target_accent in ["Nigerian", "Chinese"]:
    example = next(s for s in speakers if s["accent"] == target_accent)
    ts_path = ts_dir / f"{example['speaker_id']}.tsv"
    print(f"\n{example['speaker_id']} ({target_accent}):")
    with open(ts_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f, delimiter="\t")):
            print(f"  {row['word']:20s}  {row['start']}–{row['end']}")
            if i >= 14: break

## Stage 3 — Acoustic Distance Computation

For each target speaker × word, extracts XLS-R 300M hidden states at layer 14,
then computes DTW cosine distance against every American English reference speaker
who produced that same word. Saves `results/phonological_distance/distances/word_distances.csv`.

> **Runtime note**: ~10–20 speaker_28 on Apple MPS (with CPU fallback for weight_norm). The DTW inner loop runs on CPU; segments are short (~0.2–0.8s) so this is fast.

In [ ]:
# Rerun to regenerate; already idempotent (overwrites CSV)
!cd .. && /opt/homebrew/bin/python3.10 scripts/phonological_distance_pipeline.py --stage distances --layer 14

## Analysis

In [ ]:
df = pd.read_csv(DIST)
print(f"Loaded {len(df):,} rows")
print(df.groupby("accent")[["speaker_id","word"]].nunique())

# Reliable words: American speakers had ≥2 distinct reference speakers for that word.
# Words with n_ref==1 for American mean the only reference was the speaker themselves → distance≈0.
am_nref = df[df["accent"] == "American"].groupby("word")["n_ref"].first()
RELIABLE_WORDS = set(am_nref[am_nref >= 2].index.tolist())
excluded = sorted(set(df["word"].unique()) - RELIABLE_WORDS)
print(f"\nReliable words (n_ref≥2 for American): {len(RELIABLE_WORDS)}")
print(f"Excluded (self-only reference, unreliable baseline): {excluded}")

df_r = df[df["word"].isin(RELIABLE_WORDS)]

### Figure 1 — Mean acoustic distance from American English by accent group

In [ ]:
# Average per speaker first (so each speaker counts equally), then per accent
sp_mean = df_r.groupby(["speaker_id","accent"])["distance"].mean().reset_index()
acc_stats = (
    sp_mean.groupby("accent")["distance"]
    .agg(["mean","std","count"])
    .reindex([a for a in ACCENT_ORDER if a in sp_mean["accent"].unique()])
    .reset_index()
)

print(acc_stats.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

accents_present = acc_stats["accent"].tolist()
colors = [ACCENT_COLORS[a] for a in accents_present]
se = acc_stats["std"] / np.sqrt(acc_stats["count"])

bars = ax.bar(accents_present, acc_stats["mean"], color=colors, width=0.55,
              yerr=se * 1.96, capsize=5, edgecolor="white", linewidth=0.8)

# overlay individual speaker points
for _, row in sp_mean.iterrows():
    ax.scatter(row["accent"], row["distance"],
               color=ACCENT_COLORS.get(row["accent"],"grey"),
               alpha=0.6, zorder=5, s=30, edgecolors="black", linewidths=0.5)

ax.set_ylabel("Mean DTW cosine distance from American English", fontsize=11)
ax.set_title("Acoustic Distance from American English\n(XLS-R 300M, layer 14, personal-introduction)",
             fontsize=12)
ax.set_xlabel("Accent group", fontsize=11)
ax.set_ylim(bottom=0)
fig.tight_layout()

fig_dir = ROOT / "results" / "figures"
fig_dir.mkdir(exist_ok=True)
fig.savefig(fig_dir / "phonological_distance_by_accent.png", dpi=150)
plt.show()

### Figure 2 — Top words driving the Nigerian/Chinese distance gap

In [ ]:
# Word-level means per accent (reliable words only)
word_acc = df_r.groupby(["word","accent"])["distance"].mean().reset_index()
pivot = word_acc.pivot(index="word", columns="accent", values="distance")

if "Nigerian" in pivot.columns:
    pivot["ng_gap"] = pivot["Nigerian"] - pivot["American"]
    print("Top 15 words: Nigerian > American")
    print(pivot.dropna(subset=["ng_gap"]).nlargest(15,"ng_gap")[["Nigerian","American","ng_gap"]].round(3).to_string())

if "Chinese" in pivot.columns:
    pivot["zh_gap"] = pivot["Chinese"] - pivot["American"]
    print("\nTop 15 words: Chinese > American")
    print(pivot.dropna(subset=["zh_gap"]).nlargest(15,"zh_gap")[["Chinese","American","zh_gap"]].round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, col, label, color in [
    (axes[0], "ng_gap", "Nigerian − American", "#d62728"),
    (axes[1], "zh_gap", "Chinese − American",  "#ff7f0e"),
]:
    if col not in pivot.columns:
        ax.set_visible(False)
        continue
    top = pivot.dropna(subset=[col]).nlargest(15, col)
    ax.barh(top.index[::-1], top[col][::-1], color=color)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("DTW distance gap from American English", fontsize=10)
    ax.set_title(f"Top words: {label}", fontsize=11)

fig.suptitle("Which words drive the acoustic distance?\n(positive = target accent further from American English)",
             fontsize=12)
fig.tight_layout()
fig.savefig(fig_dir / "top_words_by_gap.png", dpi=150)
plt.show()

### Figure 3 — Distance by phonological category

Unreliable words (self-only American reference) are excluded from category membership.

In [ ]:
PHONOLOGICAL_CATEGORIES = {
    "Diphthong":         ["i've", "really", "client", "clients", "onboard", "side",
                          "options", "enjoy"],
    "Schwa/reduction":   ["about", "been", "between", "because", "their", "pretty",
                          "familiar", "understand", "customer", "customers"],
    "Consonant cluster": ["graduated", "clients", "financial", "products", "detail",
                          "services", "experience", "recently"],
    "Function word":     ["i", "a", "the", "and", "in", "to", "at", "that", "with",
                          "which", "since", "then", "most"],
    "Content word":      ["graduated", "financial", "services", "technical", "mistakes",
                          "candidate", "experience", "customers", "products",
                          "connection", "familiar", "solid", "detail"],
}

# Restrict to reliable words only
PHONOLOGICAL_CATEGORIES = {
    cat: [w for w in words if w in RELIABLE_WORDS]
    for cat, words in PHONOLOGICAL_CATEGORIES.items()
}

word_to_cat = {}
for cat, words in PHONOLOGICAL_CATEGORIES.items():
    for w in words:
        word_to_cat.setdefault(w, cat)

df_cat = df_r.copy()
df_cat["category"] = df_cat["word"].map(word_to_cat)
df_cat = df_cat.dropna(subset=["category"])

cat_acc = (
    df_cat.groupby(["accent","category"])["distance"]
    .mean().reset_index()
    .pivot(index="category", columns="accent", values="distance")
)
cat_acc = cat_acc.reindex(columns=[a for a in ACCENT_ORDER if a in cat_acc.columns])
print("Mean distance by category:")
print(cat_acc.round(3).to_string())
print("\nGap from American:")
print(cat_acc.sub(cat_acc["American"], axis=0).drop(columns="American").round(3).to_string())

In [ ]:
cats = cat_acc.index.tolist()
x = np.arange(len(cats))
n_acc = len(cat_acc.columns)
width = 0.75 / n_acc

fig, ax = plt.subplots(figsize=(12, 6))
for i, acc in enumerate(cat_acc.columns):
    offset = (i - n_acc/2 + 0.5) * width
    ax.bar(x + offset, cat_acc[acc], width=width, label=acc,
           color=ACCENT_COLORS.get(acc, "grey"), edgecolor="white", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(cats, fontsize=10)
ax.set_ylabel("Mean DTW cosine distance from American English", fontsize=10)
ax.set_title("Acoustic Distance by Phonological Category and Accent Group", fontsize=12)
ax.legend(title="Accent", loc="upper right", fontsize=9)
ax.set_ylim(bottom=0)
fig.tight_layout()
fig.savefig(fig_dir / "distance_by_phonological_category.png", dpi=150)
plt.show()

### Figure 4 — Per-word distance heatmap (top-20 highest-variance words)

In [ ]:
word_pivot = df_r.groupby(["word","accent"])["distance"].mean().reset_index()\
              .pivot(index="word", columns="accent", values="distance")
word_pivot = word_pivot.reindex(columns=[a for a in ACCENT_ORDER if a in word_pivot.columns])
word_pivot = word_pivot.dropna(thresh=3)  # need at least 3 accents

word_pivot["var"] = word_pivot.var(axis=1)
top20 = word_pivot.nlargest(20, "var").drop(columns="var")

fig, ax = plt.subplots(figsize=(8, 9))
im = ax.imshow(top20.values, aspect="auto", cmap="RdYlGn_r")
ax.set_xticks(range(len(top20.columns)))
ax.set_xticklabels(top20.columns, fontsize=10)
ax.set_yticks(range(len(top20.index)))
ax.set_yticklabels(top20.index, fontsize=9)
plt.colorbar(im, ax=ax, label="Mean DTW distance")
ax.set_title("Per-word acoustic distance from American English\n(top 20 highest-variance words across accents)",
             fontsize=11)
fig.tight_layout()
fig.savefig(fig_dir / "word_distance_heatmap.png", dpi=150)
plt.show()

### Summary statistics

In [ ]:
print("=== Speaker-level mean distances from American English ===")
print(sp_mean.groupby("accent")["distance"].describe().round(4).to_string())

print("\n=== Phonological category breakdown ===")
print(cat_acc.round(4).to_string())

print("\n=== Category gaps from American ===")
print(cat_acc.sub(cat_acc["American"], axis=0).drop(columns="American").round(4).to_string())

In [ ]:
# Word coverage per accent group
coverage = df_r.groupby("accent")["word"].nunique().sort_values(ascending=False)
print("Unique reliable script words covered per accent group:")
print(coverage.to_string())

# Words present in all 5 accent groups
shared = (
    df_r.groupby("word")["accent"].nunique()
    .pipe(lambda s: s[s == df_r["accent"].nunique()].index.tolist())
)
print(f"\nWords present across all {df_r['accent'].nunique()} accent groups ({len(shared)}):")
print(sorted(shared))